<a href="https://colab.research.google.com/github/VadimKaryakin/FractalAnalysis/blob/master/AssessDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Temp

In [1]:
#scikit 
#Adjusted Rand index
#https://scikit-learn.org/stable/modules/clustering.html

In [2]:
from sklearn import metrics
labels_true = [0, 0, 0, 1, 1, 1]
labels_pred = [0, 0, 1, 1, 2, 2]
metrics.adjusted_rand_score(labels_true, labels_pred)

0.24242424242424246

In [3]:
#One can permute 0 and 1 in the predicted labels, rename 2 to 3, and get the same score
label_pred = [1, 1, 0, 0, 3, 3]
metrics.adjusted_rand_score(labels_true, labels_pred)

0.24242424242424246

In [4]:
#independent labelings have negative or close to 0.0 score
labels_true = [0, 1, 2, 0, 3, 4, 5, 1]
labels_pred = [1, 1, 0, 0, 2, 2, 2, 2]
metrics.adjusted_rand_score(labels_true, labels_pred)

-0.12903225806451613

## Фрактальная сигнатура




In [5]:
filename = "0merged.png"
adj_scores = []
#k - количество кластеров
k = 4

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
from scipy.ndimage import generic_filter
from scipy.stats import linregress

In [7]:
def fractal_signature(imar, d_=10):
    u = imar.copy()
    b = imar.copy()

    footprint=np.array([[0, 1, 0],
                        [1, 0, 1],
                        [0, 1, 0]])
    ds = range(1, d_)
    vols = []

    for d in ds:
        fst_u = u + 1
        fst_b = b - 1

        scnd_u = maximum_filter(u, mode='constant', footprint=footprint, cval=0)
        scnd_b = minimum_filter(b, mode='constant', footprint=footprint, cval=255)

        u = np.maximum(fst_u, scnd_u)
        b = np.minimum(fst_b, scnd_b)

        vols.append(np.sum(u - b))

    x = -np.log(ds[1:d_-2])
    y = [np.log((vols[i] - vols[i-1])/2) for i in range(1, d_-2)]    
    if (len(x) != len(y)):
      print("Length error")
    return linregress(x, y).slope

In [8]:
img = cv.imread(filename, cv.IMREAD_GRAYSCALE)
print(img.shape)
#shape[0] - height, shape[1] - width
imar = np.array(img, dtype=np.int16)

(1024, 1024)


In [9]:
from scipy.ndimage.filters import maximum_filter, minimum_filter, generic_filter

In [10]:
parts = []
eps = 24

for start_y, end_y in zip(range(0, imar.shape[0]-eps, eps), range(eps, imar.shape[0], eps)):
    for start_x, end_x in zip(range(0, imar.shape[1]-eps, eps), range(eps, imar.shape[1], eps)):
        parts.append((start_x, start_y))             

In [11]:
ads26 = []
for part in parts:
  start_x, start_y = part
  ads26.append(fractal_signature(imar[start_y:start_y+eps, start_x:start_x+eps], 10))

ads10 = []
eps = 12
for part in parts:
  start_x, start_y = part
  ads10.append(fractal_signature(imar[start_y:start_y+eps, start_x:start_x+eps], 10))

In [12]:
#перевести из одного вектора во множество
if np.array(ads10)[0].size == 1:
  ads10 = np.array(ads10).reshape(-1,1)
  ads26 = np.array(ads26).reshape(-1,1)

ads = np.hstack((ads10, ads26))

##Спектр обобщенных фрактальных размерностей Реньи

In [13]:
import requests
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import multiprocessing
from io import BytesIO
from PIL import Image
from scipy.ndimage.filters import convolve
from scipy.stats import linregress

In [14]:
def rgb2gray(rgb):
    return np.dot(rgb[...,:3], [0.299, 0.587, 0.114])

In [15]:
def reni_entropy(p, q):
    return (1 / (1 - q) * np.log(np.sum(np.power(p, q)))) if q != 1 else (-np.sum(p * np.log(p)))

In [16]:
def get_reni_spectre(immat, qs, eps):
    return list(map(lambda x: get_reni_dim(immat, x, eps), qs))

In [17]:
def get_reni_dim(immat, q, eps):
    ws = range(1, eps, 3)
    ns = []
    
    for w in ws:
        conv = convolve(immat, np.ones((w, w)), mode='constant')[::w, ::w]
        ns.append(reni_entropy(conv / np.sum(conv), q))
    
    x = -np.log(ws)
    y = ns
    
    return linregress(x, y).slope

In [18]:
im = Image.open(filename)
immat = rgb2gray(np.array(im))
#чтобы исключить нулевое значение яркости
immat *= 254/255
immat += 1

In [19]:
q = np.array(range(-10, 10))
#eps - задает размер блоков arange(0, eps)
eps = 24

In [20]:
from progressbar import ProgressBar

parts = []
specs20 = []

bar = ProgressBar()

for start_y, end_y in bar(zip(range(0, immat.shape[0]-eps, eps), range(eps, immat.shape[0], eps))):
    for start_x, end_x in zip(range(0, immat.shape[1]-eps, eps), range(eps, immat.shape[1], eps)):
        parts.append((start_x, start_y))        
        specs20.append(get_reni_spectre(immat[start_y:end_y, start_x:end_x], q, eps))

| |             #                                    | 41 Elapsed Time: 0:04:58


In [21]:
from pandas import DataFrame
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn import preprocessing

specs20 = np.array(specs20)
X_scaled = preprocessing.scale(specs20)
kmeans = KMeans(n_clusters=k).fit(X_scaled)
c=kmeans.labels_.astype(int)

/usr/local/lib/python3.6/dist-packages/sklearn/preprocessing/_data.py:190: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn("Numerical issues were encountered "


##Signature + Renie



In [ ]:
from sklearn.decomposition import PCA

specs20_scaled = preprocessing.scale(specs20)
ads_scaled = preprocessing.scale(ads)

united_features = np.hstack((specs20_scaled, ads_scaled))

pca = PCA(n_components=8)
principalComponents = pca.fit_transform(united_features)

pca_norm = preprocessing.scale(principalComponents)
kmeans = KMeans(n_clusters=k).fit(pca_norm)

c=kmeans.labels_.astype(int)

##Assess


In [23]:
import math
size = math.ceil(math.sqrt(len(parts)))
labels_pred = np.zeros(len(parts)).astype(int)
patch_size = size//2

In [24]:
for i in range(0, size, 2):
  labels_pred[i*patch_size:(i+1)*patch_size] = 0
for i in range(1, size, 2):
  labels_pred[i*patch_size:(i+1)*patch_size] = 1
for i in range(size, size*2, 2):
  labels_pred[i*patch_size:(i+1)*patch_size] = 2
for i in range(size+1, size*2, 2):
  labels_pred[i*patch_size:(i+1)*patch_size] = 3

In [ ]:
#with np.printoptions(threshold=np.inf):
#  print(labels_pred)

In [27]:
adj_scores.append(metrics.adjusted_rand_score(c, labels_pred))

In [28]:
adj_scores

[0.14649196637132647]